# Optuna Merged Viewer

`optuna_jh.db`, `optuna_yi.db`, `optuna_yjs.db` 3개의 DB를 한 번에 읽어 study/trial을 병합한다.
어떤 실험(study)들이 있는지 overview를 확인한다.

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

setup 완료


## 1. `병합중/` 스캔 → `optuna_merged.db` 생성 & study summary 로드

`병합중/` 디렉토리의 모든 `optuna_*.db` 파일을 스캔해 하나의 `optuna_merged.db` 로 병합한다.

- 파일명 `optuna_<owner>_*.db` 에서 owner 를 파싱해 각 study 의 `user_attrs["owner"]` 에 주입
- **study_name 이 겹치면 `AssertionError`** (파일 내부 번호가 충돌하지 않도록 관리할 것)
- 이후 모든 셀은 `optuna_merged.db` 하나만 읽는다

In [2]:
import re
import optuna
import pandas as pd
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)

MERGE_DIR = Path("병합중")
MERGED_DB = Path("optuna_merged.db")

# ---------- 1) 병합중/ 스캔 → optuna_merged.db 재생성 ----------
assert MERGE_DIR.exists(), f"{MERGE_DIR.resolve()} 없음"
src_files = sorted(MERGE_DIR.glob("optuna_*.db"))
assert src_files, f"{MERGE_DIR} 안에 optuna_*.db 파일이 없음"

if MERGED_DB.exists():
    MERGED_DB.unlink()
merged_storage = f"sqlite:///{MERGED_DB.as_posix()}"

seen = {}  # study_name -> 첫 등장 파일
for db_path in src_files:
    m = re.match(r"optuna_([^_]+?)(?:_.+)?\.db$", db_path.name)
    owner = m.group(1) if m else "?"
    src_storage = f"sqlite:///{db_path.as_posix()}"
    for s in optuna.get_all_study_summaries(src_storage):
        sname = s.study_name
        assert sname not in seen, (
            f"study_name 중복: '{sname}' ({seen[sname]} vs {db_path.name})"
        )
        seen[sname] = db_path.name
        optuna.copy_study(
            from_study_name=sname,
            from_storage=src_storage,
            to_storage=merged_storage,
            to_study_name=sname,
        )
        # owner 를 merged DB 의 user_attrs 에 주입 (downstream 에서 사용)
        optuna.load_study(study_name=sname, storage=merged_storage).set_user_attr("owner", owner)

print(f"[merge] {len(src_files)} files → {len(seen)} studies → {MERGED_DB}")

# ---------- 2) optuna_merged.db 에서 study summary 로드 ----------
all_summaries = {}  # owner -> (storage, list[summary])
rows = []
for s in optuna.get_all_study_summaries(merged_storage):
    owner = s.user_attrs.get("owner", "?")
    if owner not in all_summaries:
        all_summaries[owner] = (merged_storage, [])
    all_summaries[owner][1].append(s)
    rows.append({
        "owner": owner,
        "study_name": s.study_name,
        "n_trials": s.n_trials,
        "direction": s.direction.name,
        "best_value": s.best_trial.value if s.best_trial else None,
    })

study_info = (
    pd.DataFrame(rows)
    .sort_values(["owner", "study_name"])
    .reset_index(drop=True)
)
print(f"[load] owners={sorted(all_summaries.keys())} | total studies={len(study_info)}")
study_info


[merge] 23 files → 29 studies → optuna_merged.db
[load] owners=['jh', 'js', 'kj', 'yi', 'yjs'] | total studies=29


,owner,study_name,n_trials,direction,best_value
0,jh,3-101-001,440,MINIMIZE,0.005544
1,jh,3-901-001,300,MINIMIZE,0.008258
2,jh,3-902-001,300,MINIMIZE,0.008631
3,jh,3-903-001,300,MINIMIZE,0.006899
4,jh,3-904-001,300,MINIMIZE,0.007369
5,jh,3-905-001,300,MINIMIZE,0.001387
6,jh,3-999-002,250,MINIMIZE,0.005521
7,js,2-101-002,300,MINIMIZE,0.001400
8,js,2-101-003,300,MINIMIZE,0.001400
9,js,2-101-004,300,MINIMIZE,0.006107


### 1-1. 제외할 study 드랍 (하드코딩)

분석에서 제외하고 싶은 study를 `(owner, study_name)` 쌍으로 지정한다.
이후 모든 셀(overview, trial 병합, 조합 매칭)은 드랍된 상태의 `study_info` / `all_summaries`를 사용한다.

In [3]:
# 제외할 study 목록 (하드코딩)
DROP_STUDIES = [
    ("yi", "1-002-001"),
    ("yi", "1-002-002"),
    ("yi", "1-003-001"),
]

_before = len(study_info)

# 1) study_info에서 드랍
drop_mask = study_info.set_index(["owner", "study_name"]).index.isin(DROP_STUDIES)
study_info = study_info[~drop_mask].reset_index(drop=True)

# 2) all_summaries에서도 동일하게 드랍 (downstream의 load_study / trials_dataframe이 제외된 상태로 돌도록)
drop_set = set(DROP_STUDIES)
for owner in list(all_summaries.keys()):
    storage, summaries = all_summaries[owner]
    filtered = [s for s in summaries if (owner, s.study_name) not in drop_set]
    all_summaries[owner] = (storage, filtered)

print(f"드랍: {_before - len(study_info)}개 / 남은 study: {len(study_info)}")
study_info

드랍: 3개 / 남은 study: 26


,owner,study_name,n_trials,direction,best_value
0,jh,3-101-001,440,MINIMIZE,0.005544
1,jh,3-901-001,300,MINIMIZE,0.008258
2,jh,3-902-001,300,MINIMIZE,0.008631
3,jh,3-903-001,300,MINIMIZE,0.006899
4,jh,3-904-001,300,MINIMIZE,0.007369
5,jh,3-905-001,300,MINIMIZE,0.001387
6,jh,3-999-002,250,MINIMIZE,0.005521
7,js,2-101-002,300,MINIMIZE,0.001400
8,js,2-101-003,300,MINIMIZE,0.001400
9,js,2-101-004,300,MINIMIZE,0.006107


## 2. Overview: owner별 실험 수 & best value

누가 어떤 study들을 돌렸는지 한눈에 본다.

In [4]:
# owner별 요약
owner_summary = (
    study_info.groupby("owner")
    .agg(
        n_studies=("study_name", "count"),
        total_trials=("n_trials", "sum"),
        best_value=("best_value", "min"),
    )
    .reset_index()
)
print("[owner별 요약]")
display(owner_summary)

# 전체 study 목록 (best_value 오름차순)
print("\n[study 목록 — best_value 오름차순]")
study_info.sort_values("best_value", na_position="last").reset_index(drop=True)

[owner별 요약]


,owner,n_studies,total_trials,best_value
0,jh,7,2190,0.001387
1,js,5,1500,0.001387
2,kj,5,1500,0.001387
3,yi,7,2231,0.001315
4,yjs,2,650,0.005456



[study 목록 — best_value 오름차순]


,owner,study_name,n_trials,direction,best_value
0,yi,1-010-001,480,MINIMIZE,0.001315
1,jh,3-905-001,300,MINIMIZE,0.001387
2,kj,5-103-001,300,MINIMIZE,0.001387
3,js,3-101-002,300,MINIMIZE,0.001387
4,yi,1-010-002,251,MINIMIZE,0.001387
5,yi,1-075-001,300,MINIMIZE,0.001387
6,js,2-101-003,300,MINIMIZE,0.001400
7,js,2-101-002,300,MINIMIZE,0.001400
8,kj,5-104-001,300,MINIMIZE,0.001400
9,yi,1-071-001,300,MINIMIZE,0.005448


## 3. 모든 trial을 하나의 DataFrame으로 병합

study_name이 owner별로 겹칠 수 있으므로 `owner`, `study_key`(= `owner/study_name`) 컬럼을 추가한다.

In [5]:
frames = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        df = study.trials_dataframe()
        if df.empty:
            continue
        df.insert(0, "owner", owner)
        df.insert(1, "study_name", s.study_name)
        df.insert(2, "study_key", f"{owner}/{s.study_name}")
        frames.append(df)

trials_df = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
print(f"전체 trial shape: {trials_df.shape}")
trials_df.head()

전체 trial shape: (8071, 66)


,owner,study_name,study_key,number,value,datetime_start,datetime_complete,duration,params_clf_colsample_bytree,params_clf_filter_threshold,...,user_attrs_clf_val_recall,user_attrs_n_feat_clean,user_attrs_n_feat_selected,user_attrs_outlier_args,user_attrs_saved_at,user_attrs_selected_cols,user_attrs_train_rmse,user_attrs_val_rmse,system_attrs_fixed_params,state
0,jh,3-101-001,jh/3-101-001,0,0.006123,2026-04-13 16:54:40.233101,2026-04-13 16:55:33.532639,0 days 00:00:53.299538,0.738578,0.25,...,0.0,569.0,570.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:55,"[X4, X6, X7, X10, X12, X14, X17, X18, X19, X21...",0.006123,0.006331,NaN,COMPLETE
1,jh,3-101-001,jh/3-101-001,1,0.006123,2026-04-13 16:55:33.690055,2026-04-13 16:56:24.209119,0 days 00:00:50.519064,0.826914,0.50,...,0.0,667.0,668.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:56,"[X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X...",0.006123,0.006331,NaN,COMPLETE
2,jh,3-101-001,jh/3-101-001,2,0.006123,2026-04-13 16:56:24.355518,2026-04-13 16:57:20.541679,0 days 00:00:56.186161,0.853378,0.45,...,0.0,663.0,664.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:57,"[X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X...",0.006123,0.006331,NaN,COMPLETE
3,jh,3-101-001,jh/3-101-001,3,0.005873,2026-04-13 16:57:20.686870,2026-04-13 16:58:43.653218,0 days 00:01:22.966348,0.848614,0.10,...,0.0,569.0,570.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:58,"[X4, X6, X7, X10, X12, X14, X17, X18, X19, X21...",0.005873,0.006085,NaN,COMPLETE
4,jh,3-101-001,jh/3-101-001,4,0.006123,2026-04-13 16:58:43.803573,2026-04-13 17:00:05.002732,0 days 00:01:21.199159,0.880904,0.10,...,0.0,671.0,672.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 17:00,"[X1, X4, X5, X6, X7, X8, X9, X10, X11, X12, X1...",0.006123,0.006331,NaN,COMPLETE


## 4. 전체 DB 통합 Top-N

3개 DB의 모든 COMPLETE trial 중 성능이 가장 좋은 trial을 뽑는다.

In [6]:
TOP_N = 20

complete = trials_df[trials_df["state"] == "COMPLETE"].copy()
# direction은 MINIMIZE 가정 (study_info에서 확인 후 다르면 수정)
top_trials = (
    complete.sort_values("value", ascending=True)
    .loc[:, ["owner", "study_name", "number", "value", "datetime_start"]]
    .head(TOP_N)
    .reset_index(drop=True)
)
top_trials

,owner,study_name,number,value,datetime_start
0,yi,1-010-001,435,0.001315,2026-04-14 04:09:46.647701
1,yi,1-010-001,419,0.001319,2026-04-14 03:37:28.889379
2,yi,1-010-001,397,0.001323,2026-04-14 02:56:42.144080
3,yi,1-010-001,383,0.001325,2026-04-14 02:29:17.623201
4,yi,1-010-001,394,0.001326,2026-04-14 02:49:18.458754
5,yi,1-010-001,352,0.001327,2026-04-14 01:26:41.045347
6,yi,1-010-001,412,0.001327,2026-04-14 03:24:36.606459
7,yi,1-010-001,381,0.001328,2026-04-14 02:25:16.581325
8,yi,1-010-001,326,0.001328,2026-04-14 00:35:04.348742
9,yi,1-010-001,321,0.001328,2026-04-14 00:24:32.652328


## 5. 특정 study만 필터링

분석에 쓸 study만 남긴다. `(owner, study_name)` 쌍으로 지정하면 정렬 순서가 바뀌어도 안전하다.

- `jh / 3-101-001`
- `yi / 1-010-001`
- `yjs / 1-100-001`

In [7]:
KEEP_STUDIES = [
    ("jh",  "3-101-001"),
    ("yi",  "1-010-001"),
    ("yjs", "1-100-001"),
]

keep_idx = pd.MultiIndex.from_tuples(KEEP_STUDIES, names=["owner", "study_name"])

study_mask = study_info.set_index(["owner", "study_name"]).index.isin(keep_idx)
study_info_sel = study_info[study_mask].reset_index(drop=True)

trial_mask = trials_df.set_index(["owner", "study_name"]).index.isin(keep_idx)
trials_sel = trials_df[trial_mask].reset_index(drop=True)

print(f"선택된 study: {len(study_info_sel)} / 전체: {len(study_info)}")
print(f"선택된 trial: {len(trials_sel)} / 전체: {len(trials_df)}")
display(study_info_sel)

선택된 study: 3 / 전체: 26
선택된 trial: 1320 / 전체: 8071


,owner,study_name,n_trials,direction,best_value
0,jh,3-101-001,440,MINIMIZE,0.005544
1,yi,1-010-001,480,MINIMIZE,0.001315
2,yjs,1-100-001,400,MINIMIZE,0.005456


## 6. 조합표(experiment_combinations.xlsx) 매칭

`experiment_combinations.xlsx`의 36개 조합 각각이 **실행됐는지 / 성능이 얼마인지**를 한눈에 본다.

**매칭 키** (study `user_attrs` 기준):

| 조합표 컬럼 | user_attrs 위치 |
|---|---|
| `run_clf` | `pipeline_config.run_clf` |
| `reg_level` | `pipeline_config.reg_level` |
| `TARGET_TRANSFORM` | `target_transform` |
| `CLIP_Y_EXTREME` | `clip_y_extreme` |
| `clf_output` | `pipeline_config.clf_output` (run_clf=False 이면 무시) |

- 한 조합에 매칭되는 study가 여러 개면 **val_rmse가 가장 낮은 것**을 채택
- 날짜는 study `datetime_start`에서 추출
- val_rmse/test_rmse/user는 study user_attrs의 `final_val_rmse`/`final_test_rmse`/`user`

In [ ]:
# 6-1. 조합표 로드 + 각 study의 config를 user_attrs에서 추출
combos = pd.read_excel("experiment_combinations.xlsx")
print(f"조합 수: {len(combos)} | 컬럼: {combos.columns.tolist()}")

cfg_rows = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        ua = dict(study.user_attrs)
        pc = ua.get("pipeline_config", {}) or {}
        # final_val_rmse가 기록 안 돼 있으면 best_trial.value로 fallback
        val_rmse = ua.get("final_val_rmse")
        if val_rmse is None and s.best_trial is not None:
            val_rmse = s.best_trial.value
        cfg_rows.append({
            "owner": owner,
            "study_name": s.study_name,
            "exp_id": ua.get("exp_id", s.study_name),
            "run_clf": pc.get("run_clf"),
            "reg_level": pc.get("reg_level"),
            "TARGET_TRANSFORM": ua.get("target_transform"),
            "CLIP_Y_EXTREME": ua.get("clip_y_extreme"),
            "clf_output": pc.get("clf_output") if pc.get("run_clf") else None,
            "n_trials": s.n_trials,
            "val_rmse": val_rmse,
            "test_rmse": ua.get("final_test_rmse"),
            "user": ua.get("user", owner),
            "날짜": s.datetime_start.date() if s.datetime_start is not None else None,
        })

study_cfg = pd.DataFrame(cfg_rows)
print(f"study config rows: {len(study_cfg)}")
study_cfg


In [ ]:
# 6-2. 조합표와 study config를 매칭 → 36행 결과표
KEY_COLS = ["run_clf", "reg_level", "TARGET_TRANSFORM", "CLIP_Y_EXTREME", "clf_output"]

# n_trials가 이 값 이상이면 "완료(O)"로 간주
DONE_TRIAL_THRESHOLD = 100

# clf_output의 NaN(= run_clf=False 일 때)은 merge가 안되므로 sentinel로 치환
NA_SENTINEL = "__NA__"
combos_m = combos[KEY_COLS].copy()
combos_m["clf_output"] = combos_m["clf_output"].fillna(NA_SENTINEL)

study_cfg_m = study_cfg.copy()
study_cfg_m["clf_output"] = study_cfg_m["clf_output"].fillna(NA_SENTINEL)

# 조합 단위로 best(val_rmse 최소) 선택
#   - val_rmse가 fallback(best_trial.value)으로도 채워지므로 dropna 불필요
#   - 그래도 전부 NaN인 study는 밀려나도록 na_position=last
best_per_combo = (
    study_cfg_m.sort_values("val_rmse", ascending=True, na_position="last")
    .drop_duplicates(subset=KEY_COLS, keep="first")
)

# 한 조합에 붙은 study 수 (val_rmse 유무 무관)
match_count = (
    study_cfg_m.groupby(KEY_COLS, dropna=False)
    .size()
    .rename("n_studies")
    .reset_index()
)

result = (
    combos_m.merge(
        best_per_combo[KEY_COLS + ["exp_id", "날짜", "n_trials", "val_rmse", "test_rmse", "user"]],
        on=KEY_COLS,
        how="left",
    )
    .merge(match_count, on=KEY_COLS, how="left")
)
result["n_studies"] = result["n_studies"].fillna(0).astype(int)
result["clf_output"] = result["clf_output"].replace(NA_SENTINEL, None)

# done 상태 3단계
#   O : n_trials >= DONE_TRIAL_THRESHOLD
#   △ : study는 있으나 trial 수가 threshold 미만
#   X : 시도 없음
def _status(row):
    if pd.notna(row["n_trials"]) and row["n_trials"] >= DONE_TRIAL_THRESHOLD:
        return "O"
    if row["n_studies"] > 0:
        return "△"
    return "X"

result["done"] = result.apply(_status, axis=1)

display_cols = KEY_COLS + ["done", "n_studies", "n_trials", "exp_id", "날짜", "val_rmse", "test_rmse", "user"]

n_done = (result["done"] == "O").sum()
n_partial = (result["done"] == "△").sum()
n_todo = (result["done"] == "X").sum()
print(f"완료(O, n_trials≥{DONE_TRIAL_THRESHOLD}): {n_done} | 진행중(△): {n_partial} | 미시도(X): {n_todo} | 총 {len(result)} 조합")

# O → △ → X 순, 동일 상태 내에서는 val_rmse 오름차순
status_order = {"O": 0, "△": 1, "X": 2}
result["_ord"] = result["done"].map(status_order)
result_sorted = (
    result.sort_values(["_ord", "val_rmse"], ascending=[True, True], na_position="last")
    .drop(columns="_ord")
    .reset_index(drop=True)
)
result_sorted[display_cols].head(30)


In [10]:
# result_sorted.to_excel("experiment_status_summary.xlsx", index=False)